# 🏢 Employee Attrition Analysis
**Objective:** Identify key drivers of employee attrition and quantify the financial cost of turnover to help HR leadership make data-driven retention decisions.

**Dataset:** IBM HR Analytics Employee Attrition (1,470 employees, 35 features)

**Tools:** Python, Pandas, Seaborn, Matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')

# Binary flag for modeling later
df['Attrition_Flag'] = (df['Attrition'] == 'Yes').astype(int)

print('Shape:', df.shape)
print('\nAttrition value counts:')
print(df['Attrition'].value_counts())
print('\nMissing values:', df.isnull().sum().sum())

---
## 1. Attrition Distribution

In [ ]:
attrition_pct = df['Attrition'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
sns.countplot(x='Attrition', data=df, ax=axes[0], palette=['#2ecc71','#e74c3c'])
axes[0].set_title('Attrition Count')
axes[0].bar_label(axes[0].containers[0], fmt='%d')

# Pie chart
axes[1].pie(attrition_pct, labels=['Stayed', 'Left'], autopct='%1.1f%%',
            colors=['#2ecc71','#e74c3c'], startangle=90)
axes[1].set_title('Attrition Rate')

plt.suptitle('Overall Attrition Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/01_attrition_distribution.png', bbox_inches='tight')
plt.show()

**Observation:** 16.1% of employees left — roughly 1 in 6.

**Business Insight:** While the count appears low, at scale this rate creates significant financial burden. The class imbalance (84/16 split) must also be handled during predictive modeling using techniques like SMOTE or class weighting.

---
## 2. Overtime vs Attrition

In [ ]:
overtime_rate = pd.crosstab(df['OverTime'], df['Attrition'], normalize='index') * 100
print(overtime_rate.round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x='OverTime', hue='Attrition', data=df, ax=axes[0],
              palette=['#2ecc71','#e74c3c'])
axes[0].set_title('Overtime vs Attrition (Count)')

overtime_rate['Yes'].plot(kind='bar', ax=axes[1], color=['#3498db','#e74c3c'],
                           edgecolor='white', width=0.5)
axes[1].set_title('Attrition RATE by Overtime (%)')
axes[1].set_ylabel('Attrition Rate (%)')
axes[1].set_xlabel('OverTime')
axes[1].axhline(y=df['Attrition_Flag'].mean()*100, color='black',
                linestyle='--', label='Company Avg')
axes[1].legend()
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.5),
                     ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('plots/02_overtime_attrition.png', bbox_inches='tight')
plt.show()

**Observation:** Employees working overtime have a ~30% attrition rate vs ~10% for those who don't — **3x higher.**

**Business Insight:** Overtime is the single strongest behavioural signal of flight risk. HR should flag employees with consistent overtime patterns for proactive check-ins and workload reviews.

---
## 3. Monthly Income vs Attrition

In [ ]:
# Salary bands
df['SalaryBand'] = pd.cut(df['MonthlyIncome'],
                           bins=[0, 3000, 6000, 10000, 20000],
                           labels=['Low (<3k)', 'Mid (3–6k)', 'High (6–10k)', 'Very High (10k+)'])

salary_rate = pd.crosstab(df['SalaryBand'], df['Attrition'], normalize='index') * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='Attrition', y='MonthlyIncome', data=df, ax=axes[0],
            palette=['#2ecc71','#e74c3c'])
axes[0].set_title('Monthly Income Distribution by Attrition')
axes[0].set_ylabel('Monthly Income ($)')

salary_rate['Yes'].plot(kind='bar', ax=axes[1], color='#e74c3c',
                         edgecolor='white', width=0.6)
axes[1].set_title('Attrition Rate by Salary Band (%)')
axes[1].set_ylabel('Attrition Rate (%)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=15)
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                     ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('plots/03_income_attrition.png', bbox_inches='tight')
plt.show()

**Observation:** Employees in the lowest salary band (<$3,000/month) have the highest attrition rate. Median income for employees who left is significantly lower than those who stayed.

**Business Insight:** Compensation is a primary retention lever. Targeted salary adjustments for high-risk, low-income employees would likely yield the highest ROI for retention spending.

---
## 4. Department-wise Attrition Rate
> ⚠️ Note: Always use **rate**, not count — departments have different sizes.

In [ ]:
dept_rate = pd.crosstab(df['Department'], df['Attrition'], normalize='index') * 100
dept_count = df.groupby('Department')['Attrition'].value_counts().unstack()

print('Attrition RATE by Department:')
print(dept_rate.round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count (misleading without context — shown for comparison)
sns.countplot(x='Department', hue='Attrition', data=df, ax=axes[0],
              palette=['#2ecc71','#e74c3c'])
axes[0].set_title('Attrition Count by Department\n(misleading — varies by dept size)')
axes[0].tick_params(axis='x', rotation=10)

# Rate (correct view)
dept_rate['Yes'].sort_values(ascending=False).plot(
    kind='bar', ax=axes[1], color=['#e74c3c','#e67e22','#3498db'],
    edgecolor='white', width=0.5)
axes[1].set_title('Attrition RATE by Department (%) ✅')
axes[1].set_ylabel('Attrition Rate (%)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=10)
axes[1].axhline(y=df['Attrition_Flag'].mean()*100, color='black',
                linestyle='--', label=f'Company Avg ({df["Attrition_Flag"].mean()*100:.1f}%)')
axes[1].legend()
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                     ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('plots/04_department_attrition.png', bbox_inches='tight')
plt.show()

**Observation:** Sales has the **highest attrition rate** (~21%), well above company average. HR has the lowest total count but a comparable rate. R&D's high count is primarily driven by its large headcount.

**Business Insight:** Sales roles are high-pressure, target-driven, and susceptible to burnout and poaching. The company should review Sales compensation structures, quota fairness, and career progression paths.

---
## 5. Job Satisfaction & Work-Life Balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Job Satisfaction rate
js_rate = pd.crosstab(df['JobSatisfaction'], df['Attrition'], normalize='index') * 100
js_rate['Yes'].plot(kind='bar', ax=axes[0], color='#e74c3c', edgecolor='white', width=0.6)
axes[0].set_title('Attrition Rate by Job Satisfaction Level')
axes[0].set_ylabel('Attrition Rate (%)')
axes[0].set_xlabel('Job Satisfaction (1=Low → 4=High)')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                     ha='center', fontsize=10)

# Work-Life Balance rate
wlb_rate = pd.crosstab(df['WorkLifeBalance'], df['Attrition'], normalize='index') * 100
wlb_rate['Yes'].plot(kind='bar', ax=axes[1], color='#9b59b6', edgecolor='white', width=0.6)
axes[1].set_title('Attrition Rate by Work-Life Balance Score')
axes[1].set_ylabel('Attrition Rate (%)')
axes[1].set_xlabel('Work-Life Balance (1=Bad → 4=Best)')
axes[1].tick_params(axis='x', rotation=0)
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                     ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('plots/05_satisfaction_wlb.png', bbox_inches='tight')
plt.show()

**Observation:** Employees with the lowest job satisfaction (Level 1) have the highest attrition rate. Similarly, employees with poor work-life balance (Level 1) are most likely to leave.

**Business Insight:** Regular pulse surveys measuring satisfaction and workload can serve as an early warning system. Proactive interventions (flexible hours, role rotation, manager coaching) for low-scoring employees could reduce flight risk significantly.

---
## 6. Tenure & Distance From Home

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Years at company
sns.histplot(data=df, x='YearsAtCompany', hue='Attrition',
             multiple='stack', palette=['#2ecc71','#e74c3c'],
             bins=20, ax=axes[0])
axes[0].set_title('Attrition by Tenure (Years at Company)')
axes[0].set_xlabel('Years at Company')
axes[0].axvline(x=2, color='black', linestyle='--', label='2-year mark')
axes[0].legend()

# Distance from home
sns.boxplot(x='Attrition', y='DistanceFromHome', data=df, ax=axes[1],
            palette=['#2ecc71','#e74c3c'])
axes[1].set_title('Distance From Home vs Attrition')
axes[1].set_ylabel('Distance From Home (km)')

plt.tight_layout()
plt.savefig('plots/06_tenure_distance.png', bbox_inches='tight')
plt.show()

**Observation:** The highest attrition occurs within the first 0–5 years, with a sharp peak in the first 2 years. Employees living farther from the office tend to leave more.

**Business Insight:** Onboarding quality and early career development have a direct impact on retention. The company should invest in structured 90-day onboarding, mentorship programmes, and regular check-ins during the critical first 2 years.

---
## 7. Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=np.number)

# Top correlations with Attrition_Flag
top_corr = numeric_df.corr()['Attrition_Flag'].drop('Attrition_Flag').sort_values()
print('Top correlations with Attrition:')
print(top_corr.tail(5).round(3))   # positively correlated
print(top_corr.head(5).round(3))   # negatively correlated

plt.figure(figsize=(16, 12))
sns.heatmap(numeric_df.corr(), cmap='coolwarm', annot=True,
            fmt='.2f', linewidths=0.5, annot_kws={'size': 7})
plt.title('Correlation Heatmap — All Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/07_correlation_heatmap.png', bbox_inches='tight')
plt.show()

---
## 8. 💰 Financial Impact Analysis
> This is the section that turns data analysis into a business case.

In [ ]:
# Industry standard: replacing an employee costs ~6 months of their salary
REPLACEMENT_MULTIPLIER = 6

left = df[df['Attrition'] == 'Yes'].copy()
stayed = df[df['Attrition'] == 'No'].copy()

total_left = len(left)
avg_salary_left = left['MonthlyIncome'].mean()
cost_per_employee = avg_salary_left * REPLACEMENT_MULTIPLIER
total_cost = cost_per_employee * total_left

# Department breakdown
dept_cost = left.groupby('Department').agg(
    employees_left=('MonthlyIncome', 'count'),
    avg_salary=('MonthlyIncome', 'mean')
)
dept_cost['replacement_cost'] = dept_cost['avg_salary'] * REPLACEMENT_MULTIPLIER * dept_cost['employees_left']

print('=' * 50)
print('        ATTRITION FINANCIAL IMPACT REPORT')
print('=' * 50)
print(f'Total employees who left:        {total_left}')
print(f'Avg monthly salary (leavers):    ${avg_salary_left:,.0f}')
print(f'Estimated cost per replacement:  ${cost_per_employee:,.0f}')
print(f'TOTAL ESTIMATED COST OF ATTRITION: ${total_cost:,.0f}')
print('=' * 50)
print('\nCost breakdown by Department:')
print(dept_cost[['employees_left', 'replacement_cost']].to_string())

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dept_cost['replacement_cost'].sort_values().plot(
    kind='barh', ax=axes[0], color='#e74c3c', edgecolor='white')
axes[0].set_title('Replacement Cost by Department ($)')
axes[0].set_xlabel('Estimated Replacement Cost ($)')
for p in axes[0].patches:
    axes[0].annotate(f'${p.get_width():,.0f}',
                     (p.get_width() + 1000, p.get_y() + 0.3), fontsize=9)

# Salary distribution: stayed vs left
axes[1].hist([stayed['MonthlyIncome'], left['MonthlyIncome']],
             bins=30, label=['Stayed', 'Left'],
             color=['#2ecc71', '#e74c3c'], alpha=0.7, edgecolor='white')
axes[1].set_title('Salary Distribution: Stayed vs Left')
axes[1].set_xlabel('Monthly Income ($)')
axes[1].legend()

plt.tight_layout()
plt.savefig('plots/08_financial_impact.png', bbox_inches='tight')
plt.show()

---
## 🔑 Key Findings Summary

| # | Finding | Implication |
|---|---------|-------------|
| 1 | **16.1% attrition rate** — 1 in 6 employees leaves | Significant ongoing cost burden |
| 2 | **Overtime employees leave 3x more** (~30% vs ~10%) | Workload management = retention lever |
| 3 | **Low salary band (<$3k/month) has highest attrition** | Targeted pay review for bottom earners |
| 4 | **Sales dept has highest attrition RATE (~21%)** | Review quotas, incentives, career path |
| 5 | **First 2 years = danger window** | Invest in structured onboarding + mentorship |
| 6 | **Low job satisfaction = strong flight signal** | Regular pulse surveys + proactive intervention |

---

## 💡 Business Recommendations

1. **Implement an overtime monitoring system** — flag employees with 3+ consecutive weeks of overtime for manager review
2. **Conduct salary benchmarking** for employees in the lowest income band, especially in Sales
3. **Redesign the first-year experience** — structured 90-day plan, buddy system, quarterly check-ins
4. **Deploy quarterly pulse surveys** to track satisfaction scores and identify at-risk teams early
5. **Build a predictive churn model** (next phase) — score every active employee monthly and prioritise retention interventions

---
*Next Phase → ML Model: Predict which current employees are most likely to leave in the next 90 days*